In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sc_control_data_final_with_agg_score_and_vuln_count.csv")

/var/folders/6f/wmsnc89s2tn1gpcjgm2rbtjr0000gn/T/ipykernel_93311/1951017037.py:4: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/sc_control_data_final_with_agg_score_and_vuln_count.csv")


In [2]:
print(df.columns)

Index(['github_repo', 'tag_name', 'tag_commit_sha', 'Binary-Artifacts_score',
       'Binary-Artifacts_reason', 'CI-Tests_score', 'CI-Tests_reason',
       'Code-Review_score', 'Code-Review_reason', 'Dangerous-Workflow_score',
       'Dangerous-Workflow_reason', 'License_score', 'License_reason',
       'Pinned-Dependencies_score', 'Pinned-Dependencies_reason',
       'Security-Policy_score', 'Security-Policy_reason',
       'Token-Permissions_score', 'Token-Permissions_reason',
       'Vulnerabilities_score', 'Vulnerabilities_reason', 'Contrib_score',
       'Contrib_reason', 'Contrib_details', 'Maintained_score',
       'Maintained_reason', 'Maintained_details', 'DependUpTool_score',
       'DependUpTool_reason', 'DependUpTool_details', 'Fuzzing_score',
       'githubrepo', 'published_at', 'project_name', 'package_name', 'System',
       'first_release_date', 'tag_name_clean', 'loc', 'dependents',
       'downloads_7_day_total', 'version_age_days', 'Aggregate_score',
       'vulnerab

In [ ]:
# --- 1. Preparation ---
# Ensure date column is in datetime format
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)

# Create a Year-Month period column (e.g., '2025-03')
df["year_month"] = df["published_at"].dt.to_period("M")

scorecard_metrics = [
    "Aggregate_score",
    "Binary-Artifacts_score",
    "Code-Review_score",    
    "Contrib_score",
    "Dangerous-Workflow_score",
    "DependUpTool_score",
    "Fuzzing_score",
    "License_score",
    "Maintained_score",
    "Pinned-Dependencies_score",
    "Security-Policy_score",
    "Token-Permissions_score"
]

# Switch -1 values to na so they are not included in mean calculations

df[scorecard_metrics] = df[scorecard_metrics].replace(-1, np.nan)

# When aggregating, keep release count as a control
monthly_panel = (
    df.groupby(["package_name", "year_month"])
    .agg(
        Vulnerability_count=("vulnerability_count", "mean"),
        Release_count=("tag_name", "count"),  # number of releases that month
        Aggregate_score=("Aggregate_score", "mean"),
        Binary_Artifacts_score=("Binary-Artifacts_score", "mean"),
        Code_Review_score=("Code-Review_score", "mean"),
        Contrib_score=("Contrib_score", "mean"),
        Dangerous_Workflow_score=("Dangerous-Workflow_score", "mean"),
        DependUpTool_score=("DependUpTool_score", "mean"),
        Fuzzing_score=("Fuzzing_score", "mean"),
        License_score=("License_score", "mean"),
        Maintained_score=("Maintained_score", "mean"),
        Pinned_Dependencies_score=("Pinned-Dependencies_score", "mean"),
        Security_Policy_score=("Security-Policy_score", "mean"),
        Token_Permissions_score=("Token-Permissions_score", "mean"),
        loc=("loc", "mean"),
        downloads_7_day_total=("downloads_7_day_total", "mean"),
        dependents=("dependents", "mean"),
        version_age_days=("version_age_days", "mean"),
    )
    .reset_index()
)

/var/folders/6f/wmsnc89s2tn1gpcjgm2rbtjr0000gn/T/ipykernel_93311/3210962606.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df["published_at"].dt.to_period("M")


In [4]:
monthly_panel.to_csv("../data/monthly_panel.csv", index=False)